### TP de clasificacion - Prediccion de lluvia en Australia
#### Aprendizaje Automático 1 — Tecnicatura en Inteligencia Artificial (FCEIA - UNR)
Integrantes: Aguirre, Gambande, Paladini

In [1]:
# Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import warnings 
warnings.filterwarnings('ignore')

# Configuración global de visualización (consistencia entre gráficos)
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100
})
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42  # semilla fija para reproducibilidad

In [111]:
df = pd.read_csv("data/weatherAUS_2026C2.csv")

### Consigna 1

### Inspeccion del dataset

In [112]:
#Revisamos el dataset para verificar tipos de datos y faltantes 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145412 entries, 0 to 145411
Data columns (total 25 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        145412 non-null  int64  
 1   Date              145412 non-null  str    
 2   Location          145412 non-null  str    
 3   MinTemp           143928 non-null  float64
 4   MaxTemp           144159 non-null  float64
 5   Rainfall          142152 non-null  float64
 6   Evaporation       82658 non-null   float64
 7   Sunshine          75616 non-null   float64
 8   WindGustDir       135096 non-null  str    
 9   WindGustSpeed     135159 non-null  float64
 10  WindDir9am        134850 non-null  str    
 11  WindDir3pm        141186 non-null  str    
 12  WindSpeed9am      143645 non-null  float64
 13  WindSpeed3pm      142351 non-null  float64
 14  Humidity9am       142759 non-null  float64
 15  Humidity3pm       140907 non-null  float64
 16  Pressure9am       130351 non-nu

In [113]:
df.columns

Index(['Unnamed: 0', 'Date', 'Location', 'MinTemp', 'MaxTemp', 'Rainfall',
       'Evaporation', 'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am',
       'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am',
       'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm',
       'Temp9am', 'Temp3pm', 'RainToday', 'RainTomorrow', 'RainfallTomorrow'],
      dtype='str')

In [114]:
df

,Unnamed: 0,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow,RainfallTomorrow
0,0,2008-12-01,Albury,13.6,22.5,0.6,NaN,NaN,W,45.0,...,23.8,1008.1,1007.2,8.0,NaN,16.8,20.8,No,No,0.0
1,1,2008-12-02,Albury,7.3,25.8,0.0,NaN,NaN,WNW,44.0,...,22.1,1011.1,1007.5,NaN,NaN,18.3,24.6,No,No,0.0
2,2,2008-12-03,Albury,13.2,26.2,0.0,NaN,NaN,WSW,46.0,...,31.5,1007.3,1009.4,NaN,2.0,21.2,23.4,No,No,0.0
3,3,2008-12-04,Albury,10.0,28.8,0.0,NaN,NaN,NE,23.0,...,18.5,1017.6,1012.4,NaN,NaN,17.4,26.3,No,No,1.0
4,4,2008-12-05,Albury,17.4,32.4,1.0,NaN,NaN,W,41.0,...,34.6,1010.9,1005.9,7.0,8.0,17.9,29.1,No,No,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145407,145454,2017-06-20,Uluru,2.6,22.0,0.0,NaN,NaN,E,29.0,...,26.5,1024.8,1021.2,NaN,NaN,10.1,21.6,No,No,0.0
145408,145455,2017-06-21,Uluru,3.1,23.2,0.0,NaN,NaN,E,31.0,...,24.7,1024.3,1020.5,NaN,NaN,11.1,21.5,No,No,0.0
145409,145456,2017-06-22,Uluru,3.6,26.0,0.0,NaN,NaN,NNW,21.0,...,21.4,1023.3,1018.4,NaN,NaN,11.8,24.5,No,No,0.0
145410,145457,2017-06-23,Uluru,4.8,27.4,0.0,NaN,NaN,N,39.0,...,23.7,1021.0,1016.0,NaN,NaN,12.0,26.9,No,No,0.0


### 1.1 Análisis y decisión sobre datos faltantes

In [115]:
# Observamos que la columna Date se encontraba como tipo str. Ante esto, se decide convertirla a datetime.
df["Date"] = pd.to_datetime(df["Date"])

In [116]:
# Se decide elimininar la columna 'Unnamed: 0' ya que solo representaba indices dentro del dataset.
df = df.drop(columns = ['Unnamed: 0'])

In [117]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145412 entries, 0 to 145411
Data columns (total 24 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Date              145412 non-null  datetime64[us]
 1   Location          145412 non-null  str           
 2   MinTemp           143928 non-null  float64       
 3   MaxTemp           144159 non-null  float64       
 4   Rainfall          142152 non-null  float64       
 5   Evaporation       82658 non-null   float64       
 6   Sunshine          75616 non-null   float64       
 7   WindGustDir       135096 non-null  str           
 8   WindGustSpeed     135159 non-null  float64       
 9   WindDir9am        134850 non-null  str           
 10  WindDir3pm        141186 non-null  str           
 11  WindSpeed9am      143645 non-null  float64       
 12  WindSpeed3pm      142351 non-null  float64       
 13  Humidity9am       142759 non-null  float64       
 14  Humidity3pm    

In [118]:
# Mostramos el porcentaje de nulos de cada variable respecto al total y la cantidad de faltantes en cada variable
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df)) * 100

reporte_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje_nulos.round(2)})
reporte_nulos = reporte_nulos[reporte_nulos['Nulos'] > 0].sort_values('Nulos', ascending=False)
reporte_nulos

,Nulos,Porcentaje (%)
Sunshine,69796,48.00
Evaporation,62754,43.16
Cloud3pm,59336,40.81
Cloud9am,55870,38.42
Pressure9am,15061,10.36
Pressure3pm,15024,10.33
WindDir9am,10562,7.26
WindGustDir,10316,7.09
WindGustSpeed,10253,7.05
Humidity3pm,4505,3.10


In [119]:
# Distribución de filas según cantidad y porcentaje de columnas faltantes
nulos_por_fila = df.isnull().sum(axis=1)
dist = nulos_por_fila.value_counts().sort_index().to_frame('Filas')
dist['% columnas faltantes'] = (dist.index / df.shape[1] * 100).round(1)
dist

,Filas,% columnas faltantes
0,56412,0.0
1,10764,4.2
2,18167,8.3
3,9920,12.5
4,28702,16.7
5,3774,20.8
6,10432,25.0
7,2087,29.2
8,1440,33.3
9,500,37.5


Se observa que la variable target (RainTomorrow) posee 3259 valores faltantes, lo que equivale aproximadamente a 2.24% de los valores faltantes. Como imputar la variable target implicaria tener datos artificiales en nuestra prediccion, se decide eliminar esas filas en lugar de imputarlas.

In [124]:
df = df.dropna(subset= ["RainTomorrow"])

Volvemos a recalcular los faltantes y porcentajes por variable para actualizar los datos de cada una de las mismas

In [125]:
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df)) * 100

reporte_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje_nulos.round(2)})
reporte_nulos = reporte_nulos[reporte_nulos['Nulos'] > 0].sort_values('Nulos', ascending=False)
reporte_nulos

,Nulos,Porcentaje (%)
Sunshine,67785,47.68
Evaporation,60812,42.78
Cloud3pm,57076,40.15
Cloud9am,53641,37.73
Pressure9am,14010,9.86
Pressure3pm,13977,9.83
WindDir9am,10009,7.04
WindGustDir,9328,6.56
WindGustSpeed,9268,6.52
WindDir3pm,3776,2.66


In [129]:
nulos_por_fila = df.isnull().sum(axis=1)
dist = nulos_por_fila.value_counts().sort_index().to_frame('Filas')
dist['% columnas faltantes'] = (dist.index / df.shape[1] * 100).round(1)
dist

,Filas,% columnas faltantes
0,56412,0.0
1,10764,4.2
2,18023,8.3
3,9607,12.5
4,28531,16.7
5,3639,20.8
6,9934,25.0
7,1801,29.2
8,964,33.3
9,393,37.5


In [154]:
df.info()

<class 'pandas.DataFrame'>
Index: 142153 entries, 0 to 145411
Data columns (total 24 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Date              142153 non-null  datetime64[us]
 1   Location          142153 non-null  str           
 2   MinTemp           141516 non-null  float64       
 3   MaxTemp           141831 non-null  float64       
 4   Rainfall          140747 non-null  float64       
 5   Evaporation       81341 non-null   float64       
 6   Sunshine          74368 non-null   float64       
 7   WindGustDir       132825 non-null  str           
 8   WindGustSpeed     132885 non-null  float64       
 9   WindDir9am        132144 non-null  str           
 10  WindDir3pm        138377 non-null  str           
 11  WindSpeed9am      140805 non-null  float64       
 12  WindSpeed3pm      139524 non-null  float64       
 13  Humidity9am       140379 non-null  float64       
 14  Humidity3pm       13

Realizamos la division de variables por numericas, categoricas y la target

In [161]:
target = df["RainTomorrow"]

variables_numericas = df.select_dtypes(include="number").columns.tolist()
variables_categoricas = df.select_dtypes(include="object").columns.tolist()

# Sacamos la target de categóricas
variables_categoricas.remove("RainTomorrow")

In [162]:
print(variables_numericas)

['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainfallTomorrow']


In [167]:
descripcion = df[variables_numericas].describe().T
descripcion["sesgo"] = df[variables_numericas].skew()
descripcion["curtosis"] = df[variables_numericas].kurtosis()

In [168]:
descripcion

,count,mean,std,min,25%,50%,75%,max,sesgo,curtosis
MinTemp,141516.0,12.188866,6.422856,-9.7,7.6,12.0,16.8,34.1,0.022221,-0.481609
MaxTemp,141831.0,23.228596,7.134436,-5.1,17.9,22.6,28.2,49.1,0.222679,-0.235140
Rainfall,140747.0,2.350461,8.466267,0.0,0.0,0.0,0.8,371.0,9.886866,179.957216
Evaporation,81341.0,5.470149,4.188649,0.0,2.6,4.8,7.4,145.0,3.746758,45.066745
Sunshine,74368.0,7.624919,3.781656,0.0,4.9,8.5,10.6,14.5,-0.502936,-0.820411
WindGustSpeed,132885.0,39.985807,13.630314,6.0,31.0,38.0,48.0,135.0,0.866666,1.401362
WindSpeed9am,140805.0,14.025219,8.914326,0.0,7.0,13.0,19.0,130.0,0.785296,1.220041
WindSpeed3pm,139524.0,18.636901,8.858849,0.0,12.0,18.0,24.0,86.0,0.619921,0.741008
Humidity9am,140379.0,68.824619,19.085026,0.0,56.9,69.9,83.1,100.0,-0.484886,-0.044226
Humidity3pm,138544.0,51.475886,20.846540,0.0,36.5,51.9,65.6,100.0,0.032263,-0.509691
